# 🔐 [강사용] 8주차 과제 — 정답/레퍼런스 (난이도 조정판)

> 이 노트북은 **정답 예시 파일 생성용**입니다. 실습 환경에 맞게 일부 값(이름/이메일/시간)은 바꿔도 됩니다.


In [ ]:
# 0) 레퍼런스 제출물 생성
import json, csv, random, re, math
from pathlib import Path
from datetime import datetime, timedelta, timezone

REF = Path("./reference/submit")
REF.mkdir(parents=True, exist_ok=True)

print("✅ reference/submit 폴더 준비 완료")


## Q1 레퍼런스 workflow_q1.json 생성

- Open-Meteo API 사용
- 15분 주기
- Set 1개 매핑
- 표현식 포함


In [ ]:
Q1_REFERENCE = {
  "name": "[홍길동]_Weather_Alert",
  "nodes": [
    {
      "parameters": {
        "rule": {
          "interval": [
            {
              "field": "minutes",
              "minutesInterval": 15
            }
          ]
        }
      },
      "id": "n1",
      "name": "Schedule",
      "type": "n8n-nodes-base.scheduleTrigger",
      "typeVersion": 1.1,
      "position": [
        220,
        300
      ]
    },
    {
      "parameters": {
        "url": "https://api.open-meteo.com/v1/forecast",
        "method": "GET",
        "sendQuery": true,
        "queryParameters": {
          "parameters": [
            {
              "name": "latitude",
              "value": "37.5665"
            },
            {
              "name": "longitude",
              "value": "126.9780"
            },
            {
              "name": "current",
              "value": "temperature_2m,precipitation"
            }
          ]
        },
        "options": {}
      },
      "id": "n2",
      "name": "Weather API",
      "type": "n8n-nodes-base.httpRequest",
      "typeVersion": 4.2,
      "position": [
        440,
        300
      ]
    },
    {
      "parameters": {
        "assignments": {
          "assignments": [
            {
              "id": "a1",
              "name": "temp_c",
              "value": "={{ $json.current.temperature_2m }}",
              "type": "number"
            }
          ]
        },
        "options": {}
      },
      "id": "n3",
      "name": "Shape",
      "type": "n8n-nodes-base.set",
      "typeVersion": 3.4,
      "position": [
        660,
        300
      ]
    },
    {
      "parameters": {
        "conditions": {
          "options": {
            "caseSensitive": true,
            "typeValidation": "strict"
          },
          "conditions": [
            {
              "leftValue": "={{ $json.temp_c }}",
              "rightValue": 30,
              "operator": {
                "type": "number",
                "operation": "gte"
              }
            }
          ]
        }
      },
      "id": "n4",
      "name": "IF temp>=30",
      "type": "n8n-nodes-base.if",
      "typeVersion": 2.2,
      "position": [
        880,
        300
      ]
    },
    {
      "parameters": {},
      "id": "n5",
      "name": "Alert (NoOp)",
      "type": "n8n-nodes-base.noOp",
      "typeVersion": 1,
      "position": [
        1100,
        220
      ]
    },
    {
      "parameters": {},
      "id": "n6",
      "name": "Normal (NoOp)",
      "type": "n8n-nodes-base.noOp",
      "typeVersion": 1,
      "position": [
        1100,
        380
      ]
    }
  ],
  "connections": {
    "Schedule": {
      "main": [
        [
          {
            "node": "Weather API",
            "type": "main",
            "index": 0
          }
        ]
      ]
    },
    "Weather API": {
      "main": [
        [
          {
            "node": "Shape",
            "type": "main",
            "index": 0
          }
        ]
      ]
    },
    "Shape": {
      "main": [
        [
          {
            "node": "IF temp>=30",
            "type": "main",
            "index": 0
          }
        ]
      ]
    },
    "IF temp>=30": {
      "main": [
        [
          {
            "node": "Alert (NoOp)",
            "type": "main",
            "index": 0
          }
        ],
        [
          {
            "node": "Normal (NoOp)",
            "type": "main",
            "index": 0
          }
        ]
      ]
    }
  },
  "pinData": {},
  "active": false,
  "versionId": "v1-sample",
  "meta": {
    "instanceId": "sample"
  }
}
(REF / "workflow_q1.json").write_text(json.dumps(Q1_REFERENCE, ensure_ascii=False, indent=2), encoding="utf-8")
print("✅ Q1 레퍼런스 생성:", REF/"workflow_q1.json")


## Q2 레퍼런스 openclaw_setup.json / gmail_log.jsonl / gmail_summary.md

- **authorized_email** 은 반드시 학생 본인 이메일로 교체
- 로그는 8건(완화판)


In [ ]:
OPENCLAW = {
    "openclaw_version": "0.3.1",
    "node_version": "v20.11.0",
    "oauth_token_path": "/home/student/.openclaw/token.json",
    "scopes": ["https://www.googleapis.com/auth/gmail.readonly"],
    "authorized_email": "honggildong@gmail.com",  # TODO: 본인 이메일로 바꾸세요
    "authorized_at": "2026-04-19T10:00:00Z",
}
(REF / "openclaw_setup.json").write_text(json.dumps(OPENCLAW, ensure_ascii=False, indent=2), encoding="utf-8")

random.seed(7)
def gid(n=16): return "".join(random.choice("0123456789abcdef") for _ in range(n))

def mask_email(e):
    if "@" not in e: return "***"
    lo, do = e.split("@", 1)
    return f"{lo[:2]}***@{do}" if len(lo) > 2 else f"{lo[0]}***@{do}"

senders = ["notifications@coursera.org","team@github.com","noreply@linkedin.com","newsletter@medium.com",
           "alerts@google.com","hello@figma.com","billing@stripe.com","updates@notion.so"]
subjects = ["Course update","PR review requested","Weekly digest","Article stats",
            "Security notification","New comment","Invoice available","Workspace update"]

now = datetime.now(timezone.utc)
to_masked = mask_email(OPENCLAW["authorized_email"])

with (REF/"gmail_log.jsonl").open("w", encoding="utf-8") as f:
    for i in range(8):
        mid = gid(16)
        row = {
            "message_id": mid,
            "thread_id": mid,
            "from_masked": mask_email(senders[i]),
            "to_masked": to_masked,
            "subject": subjects[i],
            "snippet_masked": f"Snippet {i+1} ... [이메일] ...",
            "received_at": (now - timedelta(days=i)).strftime("%Y-%m-%dT%H:%M:%SZ"),
        }
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

(REF/"gmail_summary.md").write_text(
    "# Gmail 요약\n\n최근 7일 메일 8건을 확인했습니다. "
    "대부분 서비스 알림(코스, 코드리뷰, 보안 알림, 결제/워크스페이스 업데이트) 성격이며 "
    "업무/학습 진행 상황을 알려주는 메시지가 중심입니다. "
    "주요 도메인은 coursera.org, github.com, google.com, notion.so 등이며 "
    "개인정보 노출을 막기 위해 이메일/토큰은 마스킹 처리했습니다.\n",
    encoding="utf-8"
)

print("✅ Q2 레퍼런스 생성:", REF/"openclaw_setup.json", REF/"gmail_log.jsonl", REF/"gmail_summary.md")


## Q3 레퍼런스: harness.py / q3_eval_set.jsonl / harness_report.json / experiment_matrix.csv

- 4개 메트릭 직접 구현
- dummy retriever/llm 기반
- 2개 실험(exp1/exp2) 점수 차이가 나도록 설계


In [ ]:
# harness.py
(REF/"harness.py").write_text(HARNESS_PY_REF, encoding="utf-8")

# eval_set (in:7 / out:3)
eval_set = [
    {"q_id":1,"question":"n8n Schedule Trigger의 역할은?","expected_answer":"주기적으로 워크플로우를 실행한다",
     "ground_truth_chunk_ids":["c_001"],"scope":"in"},
    {"q_id":2,"question":"HTTP Request 노드는 무엇을 하나?","expected_answer":"외부 API를 호출해 데이터를 가져온다",
     "ground_truth_chunk_ids":["c_001"],"scope":"in"},
    {"q_id":3,"question":"Set 노드는 왜 쓰나?","expected_answer":"필드를 정리/매핑해 다음 노드로 전달한다",
     "ground_truth_chunk_ids":["c_001"],"scope":"in"},
    {"q_id":4,"question":"IF 노드의 브랜치는?","expected_answer":"true/false로 분기한다",
     "ground_truth_chunk_ids":["c_001"],"scope":"in"},
    {"q_id":5,"question":"n8n 표현식 예시는?","expected_answer":"={{ $json.field }}",
     "ground_truth_chunk_ids":["c_001"],"scope":"in"},
    {"q_id":6,"question":"RAG의 목표는?","expected_answer":"할루시네이션 완화와 최신/외부 지식 보강",
     "ground_truth_chunk_ids":["c_001"],"scope":"in"},
    {"q_id":7,"question":"Context Recall은 무엇을 측정?","expected_answer":"정답 근거 chunk가 검색 결과에 포함되는지",
     "ground_truth_chunk_ids":["c_001"],"scope":"in"},
    {"q_id":8,"question":"2026년 매출 목표는?","expected_answer":"문서에 없음",
     "ground_truth_chunk_ids":[],"scope":"out"},
    {"q_id":9,"question":"강사 생년월일은?","expected_answer":"문서에 없음",
     "ground_truth_chunk_ids":[],"scope":"out"},
    {"q_id":10,"question":"n8n 차기 버전 일정?","expected_answer":"문서에 없음",
     "ground_truth_chunk_ids":[],"scope":"out"},
]
with (REF/"q3_eval_set.jsonl").open("w", encoding="utf-8") as f:
    for r in eval_set:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

# harness_report.json / experiment_matrix.csv 는 harness.py 실행으로 생성하는 형태라,
# 여기서는 레퍼런스용으로 한번 직접 실행 가능한 형태로만 둡니다.
print("✅ Q3 레퍼런스 생성:", REF/"harness.py", REF/"q3_eval_set.jsonl")
print("   - harness_report.json / experiment_matrix.csv 는 harness.py 실행으로 생성")


### 실행(정답 검증)

```bash
# (프로젝트 루트에서)
cp -r reference/submit ./submit
python submit/harness.py
```

생성된 `submit/harness_report.json`, `submit/experiment_matrix.csv` 를 확인하세요.
